# M5 · Offline metrics

_AFP-AI · Domain 0 · ML Foundations_

**Compute classification and ranking metrics from scored examples.**

We calculate AUC and NDCG directly, then compare against scikit-learn. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

rng = np.random.default_rng(5)

## Key formulas

Precision is $\frac{TP}{TP+FP}$, recall is $\frac{TP}{TP+FN}$, and

$$NDCG@k=\frac{\sum_{i=1}^k rel_i / \log_2(i+1)}{IDCG@k}$$

In [ ]:
scores = np.array([0.92, 0.81, 0.55, 0.44, 0.20, 0.10])
labels = np.array([1, 0, 1, 0, 1, 0])
df = pd.DataFrame({"score": scores, "label": labels})

df

## Step 1 - Compute AUC by pair counting

AUC counts how often a positive score beats a negative score.

In [ ]:
positive_scores = scores[labels == 1]
negative_scores = scores[labels == 0]
wins = 0.0
pairs = 0

for ps in positive_scores:
    for ns in negative_scores:
        wins = wins + float(ps > ns)
        wins = wins + 0.5 * float(ps == ns)
        pairs = pairs + 1

auc_manual = wins / pairs
auc_sklearn = roc_auc_score(labels, scores)

print("manual AUC", auc_manual)
print("sklearn AUC", auc_sklearn)

assert abs(auc_manual - auc_sklearn) < 1e-12

## Step 2 - Compute threshold metrics

Precision and recall require converting scores to decisions.

In [ ]:
threshold = 0.5
pred = (scores >= threshold).astype(int)
precision, recall, f1, support = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)

print("precision", round(precision, 3))
print("recall", round(recall, 3))
print("f1", round(f1, 3))

assert round(precision, 3) == 0.667

## Step 3 - Compute NDCG@5

Now use graded relevance in displayed rank order.

In [ ]:
relevance = np.array([3.0, 0.0, 2.0, 1.0, 0.0])
discounts = np.log2(np.arange(2, len(relevance) + 2))
dcg = np.sum(relevance / discounts)
ideal = np.sort(relevance)[::-1]
idcg = np.sum(ideal / discounts)
ndcg = dcg / idcg

print("DCG", round(dcg, 3))
print("IDCG", round(idcg, 3))
print("NDCG", round(ndcg, 3))

assert 0.0 <= ndcg <= 1.0

## Visualize scores and labels

Metrics summarize this ranked list; the plot lets you inspect the ordering directly.

In [ ]:
colors = np.where(labels == 1, "#54a24b", "#e45756")
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(np.arange(len(scores)), scores, color=colors)
ax.axhline(threshold, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("ranked item")
ax.set_ylabel("score")
ax.set_title("scores, labels, and a threshold")
plt.show()

## Practice

1. Change the threshold to 0.8 and recompute precision and recall.
2. Swap the top two relevance values and recompute NDCG.
3. Create a slice with the first three rows and compute its AUC.

In [ ]:
# Your turn:
